In [1]:
import src.database.scripts.sql as sql 

import requests
from datetime import datetime, timedelta, timezone
import numpy as np

In [2]:
class recipe_fetch():
    def __init__(self):
        self.conn = sql.connect_pc()
        self.cursor = self.conn.cursor()


    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.conn.close()
        self.conn = None

    def __del__(self):
        if self.conn:
            self.conn.close()



    def id_range(self):
        query = """
        SELECT recipe_id FROM recipes
        WHERE name LIKE '%Charm%'
        OR name LIKE '%Vial%'
        OR name LIKE '%Ingot%'
        OR name LIKE '%Powder%'
        OR name LIKE '%Vial%'
        OR name LIKE '%Ghostdust%'
        OR name LIKE '%Gold Coin%'
        OR name LIKE '%Potion%'
        OR name LIKE '%Fangs%'
        OR name LIKE '%Campfire%'
        OR name LIKE '%Woodsman%'
        OR name LIKE '%Intricate%'
        OR name LIKE '%LockPicks%'
        """
        # query = """
        # SELECT recipe_id FROM recipes
        # """
        self.cursor.execute(query)
        return self.cursor.fetchall()

    def craftable(self, recipe_id):
        query = f"""
        SELECT amount, rarity, name FROM recipes
        WHERE recipe_id = {recipe_id}
        """
        self.cursor.execute(query)
        return self.cursor.fetchall() 

    def ingredients(self, recipe_id):
        query = f"""
        SELECT amount, rarity, name FROM ingredients
        WHERE recipe_id = {recipe_id}
        """
        self.cursor.execute(query)
        return self.cursor.fetchall()



In [3]:
class darkerdb():
    def __init__(self):
        self.ses = requests.Session()

    def get_url(self, name, rarity):
        from_date = (datetime.now(timezone.utc) - timedelta(hours=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
        self.url = f"https://api.darkerdb.com/v1/market?item={name.replace('\'', "’")}&rarity={rarity}&from={from_date}&limit=50&has_sold=0"
    def price_fetch(self):
        fetch = self.ses.get(self.url, timeout=10)
        price = tuple(item['price_per_unit'] for item in fetch.json()['body'])
        return (price)

In [12]:
db = darkerdb()


with recipe_fetch() as recipe:
    output = [('recipe_id', 'rarity', 'name', 'net', 'r_avg', 'i_avg', 'q_sold')]    
    for item in recipe.id_range():
        r_avg = 0
        i_avg = 0
        # print(item)
        
        for r_amount, r_rarity, r_name in recipe.craftable(item[0]):
            print(r_name, end="\r", flush=True)
            db.get_url(r_name, r_rarity)
            price_list = db.price_fetch()
            q_sold = len(price_list)
            if price_list == (): pass
            else: r_avg += round(np.nanmean(price_list)) * r_amount

        for i_amount, i_rarity, i_name in recipe.ingredients(item[0]):
            if i_name != 'Gold Coin':
                db.get_url(i_name, i_rarity)
                price_list = db.price_fetch()
                if not price_list: pass
                else: i_avg += round(np.nanmean(price_list)) * i_amount
            else:
                i_avg += i_amount

        output.append((item[0], r_rarity, r_name, r_avg-i_avg, r_avg, i_avg, q_sold))
        # print(f"{item[0]} {r_rarity:<10}  {r_name:<35} net:{(r_avg-i_avg):>6}  r:{r_avg:>4}  i:{i_avg:>5}  q:{amount_sold:>5}")


In [13]:
output[1:] = sorted(output[1:], key=lambda x: x[3], reverse=True)
for x in output:
    print(f"{x[0]:<10} {x[1]:<10} {x[2]:<35} {(x[3]):>8} {x[4]:>8} {x[5]:>8} {x[6]:>8}")

recipe_id  rarity     name                                     net    r_avg    i_avg   q_sold
6          Epic       Gold Powder                             3992     4396      404        7
12         Epic       Potion of Healing                       1140     1773      633       50
18         Uncommon   Magic Protection Potion                  753      936      183       50
23         Rare       Potion of Clarity                        574     1464      890       40
22         Uncommon   Potion of Clarity                        302      777      475       50
25         Uncommon   Potion of Water Breathing                282      282        0        3
15         Rare       Potion of Protection                     275      879      604       50
11         Rare       Potion of Healing                        211      678      467       50
16         Epic       Potion of Protection                     209     1305     1096       50
26         Rare       Potion of Water Breathing             

In [11]:
r = recipe_fetch()
db = darkerdb()
for amount, rarity, name in r.ingredients(109):
        # print(amount, rarity, name)
        db.get_url(name, rarity)
        price_list = db.price_fetch()
        if name != 'Gold Coin':
                print(f"{name} ({round(np.nanmean(price_list)) * amount}) {price_list}")
        else: print(f"{name} ({amount})")

Bone Powder (26515) (52631, 399)
Wolf Fang (430) (199, 200, 262.33, 200)
Wolf Claw (264) (122, 55)
Gold Coin (300)


In [9]:
db = darkerdb()
db.get_url('Gold Coin Chest','Unique')
fetch = db.price_fetch()
fetch

(9777,
 9777,
 7000,
 8000,
 9800,
 9888,
 12222,
 9999,
 9876,
 9750,
 9799,
 9500,
 9700,
 9600,
 9750,
 8399,
 8999,
 8999,
 8999,
 8999,
 8999,
 8999,
 7878,
 9000,
 9000,
 9000,
 9000,
 9400,
 9500,
 9800,
 8000,
 9900,
 9995,
 9155,
 9555,
 9222,
 9996,
 9996,
 9996,
 9997,
 7500,
 7500,
 9900,
 8000,
 9998,
 9999,
 9500,
 8999,
 8555,
 10000)